# Experiments
This file contains multiple experiments that were done in order to find the solution to the [task](../task.md). Most of the necesary code is located in [piatek_Biegacz_Cieslik.py](piatek_Biegacz_Cieslik.py).

### Setup and environment check

In [ ]:
# %pip install numpy
# %pip install torch
# %pip install torchvision
# %pip install pytorch-fid

In [ ]:
from pytorch_fid import fid_score
import os, sys, subprocess, shutil
import numpy as np, pandas as pd
import torch, torchvision
from torch.utils.data import Subset
from piatek_Biegacz_Cieslik import (
    VAEConfig, 
    VAEExperiment, 
    read_train_dataset, 
    create_data_loader, 
    denormalize_images
)

# Configuration & Environment
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
FLAT_REFERENCE_DIR = "temp_reference_flat"

### FID Evaluation

In [ ]:
from pytorch_fid import fid_score

def compute_fid(generated_images, reference_dir):
    """
    Saves generated samples as PNG files, then uses pytorch_fid Python API to compute FID.
    Uses num_workers=0 to avoid multiprocessing issues on Windows.
    """
    temp_gen_dir = "temp_generated_samples"
    if os.path.exists(temp_gen_dir):
        shutil.rmtree(temp_gen_dir)
    os.makedirs(temp_gen_dir)
    
    for i in range(len(generated_images)):
        torchvision.utils.save_image(generated_images[i], os.path.join(temp_gen_dir, f"{i:04d}.png"))
    
    try:
        fid_value = fid_score.calculate_fid_given_paths(
            [temp_gen_dir, reference_dir],
            batch_size=64,
            device=DEVICE,
            dims=2048,
            num_workers=0  # Avoid Windows multiprocessing PermissionError
        )
    except Exception as e:
        print(f"FID computation failed: {e}")
        fid_value = 1000.0
    finally:
        shutil.rmtree(temp_gen_dir)
    
    return fid_value

def run_and_evaluate_configuration(experiment_name, config, training_set):
    """
    Handles the full pipeline for a single experiment: training, generation, and FID scoring.
    """
    subset_indices = np.random.choice(len(training_set), size=int(0.8 * len(training_set)), replace=False)
    data_loader = create_data_loader(Subset(training_set, subset_indices), config)
    
    vae_experiment = VAEExperiment(config)
    vae_experiment.fit(data_loader, print_progress=False)
    
    generated_samples = denormalize_images(vae_experiment.generate_samples(1000), config)
    fid_score = compute_fid(generated_samples, FLAT_REFERENCE_DIR)
    
    result = {"name": experiment_name, "fid": fid_score}
    result.update({k: v for k, v in config.__dict__.items() if k != 'train_data_path'})
    return result

def display_results(results, title="Experiment Results"):
    """Renders a styled DataFrame in the notebook with FID color gradient."""
    cols = ["name", "fid", "latent_dimension", "encoder_channels", "batch_size", "learning_rate", "kl_weight", "epochs"]
    df = pd.DataFrame(results).sort_values("fid").reset_index(drop=True)
    df = df[[c for c in cols if c in df.columns]]
    styled = (
        df.style
        .background_gradient(subset=["fid"], cmap="RdYlGn_r")
        .format({"fid": "{:.2f}", "learning_rate": "{:.1e}", "kl_weight": "{:.2f}"})
        .set_caption(title)
        .set_properties(**{"text-align": "center"})
        .set_table_styles([{"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold"), ("padding", "8px")]}])
        .highlight_min(subset=["fid"], props="font-weight: bold; border: 2px solid #2ecc71;")
    )
    display(styled)

Load data and prepare a flattened directory for pytorch-fid (required for nested datasets)


In [ ]:
base_config = VAEConfig()
_, train_dataset = read_train_dataset(base_config)

print(f"Searching for images in: {base_config.train_data_path}")

if not os.path.exists(FLAT_REFERENCE_DIR) or not os.listdir(FLAT_REFERENCE_DIR):
    if not os.path.exists(FLAT_REFERENCE_DIR):
        os.makedirs(FLAT_REFERENCE_DIR)
    
    print("Creating flat reference view (this may take a moment)...")
    count = 0
    image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.ppm', '.webp', '.tiff')
    
    for root, _, files in os.walk(base_config.train_data_path):
        for file in files:
            if file.lower().endswith(image_extensions):
                source = os.path.abspath(os.path.join(root, file))
                target = os.path.join(FLAT_REFERENCE_DIR, f"ref_{count:06d}_{file}")
                try:
                    os.symlink(source, target)
                except OSError:
                    # Fallback for Windows without developer mode
                    shutil.copy(source, target)
                count += 1
                
print(f"Reference ready with {len(os.listdir(FLAT_REFERENCE_DIR))} images.")

### Experiments

##### Latent Space Dimensionality

In [ ]:
all_experiment_results0 = []

for dim in [32, 64, 128, 256, 512]:
    config = VAEConfig(latent_dimension=dim, epochs=1)
    all_experiment_results0.append(run_and_evaluate_configuration(f"Latent_{dim}", config, train_dataset))
display_results(all_experiment_results0, "Latent Space Dimensionality")

In [ ]:
all_experiment_results = []

for dim in [256, 512, 1024, 2048]:
    config = VAEConfig(latent_dimension=dim, epochs=50, kl_weight=0.01)
    all_experiment_results.append(run_and_evaluate_configuration(f"Latent_{dim}", config, train_dataset))
display_results(all_experiment_results, "Latent Space Dimensionality")

100%|██████████| 613/613 [01:38<00:00,  6.20it/s]


,name,fid,latent_dimension,encoder_channels,batch_size,learning_rate,kl_weight,epochs
0,Latent_1024,170.55,1024,"[32, 64, 128]",128,1.0e-03,0.01,30
1,Latent_2048,171.74,2048,"[32, 64, 128]",128,1.0e-03,0.01,30
2,Latent_512,176.10,512,"[32, 64, 128]",128,1.0e-03,0.01,30
3,Latent_256,180.40,256,"[32, 64, 128]",128,1.0e-03,0.01,30


#### ?

In [ ]:
kl_results = []
for kl in [0.01, 0.05, 0.1, 0.5]:
    config = VAEConfig(kl_weight=kl, latent_dimension=1024, epochs=50, batch_size=32, encoder_channels=[16, 32, 64],)
    kl_results.append(run_and_evaluate_configuration(f"KL: {kl}", config, train_dataset))

all_experiment_results.append(kl_results)
display_results(kl_results, "+ KL")

100%|██████████| 613/613 [01:34<00:00,  6.48it/s]


,name,fid,latent_dimension,encoder_channels,batch_size,learning_rate,kl_weight,epochs
0,KL: 0.01,167.73,1024,"[32, 64, 128]",32,1.0e-03,0.01,40
1,KL: 0.05,181.93,1024,"[32, 64, 128]",32,1.0e-03,0.05,40
2,KL: 0.1,197.86,1024,"[32, 64, 128]",32,1.0e-03,0.10,40
3,KL: 0.5,275.45,1024,"[32, 64, 128]",32,1.0e-03,0.50,40


##### Network Depth and Capacity

In [ ]:
architectures = {
    "64_128":     [64, 128],          
    "128_256":    [128, 256],         
    "64_128_256":  [64, 128, 256],     
    "32_64_128_256":     [32, 64, 128, 256],     
    "16_32_64":      [16, 32, 64],        
    "16_32_64_128":        [16, 32, 64, 128],        
    "64_64_128":    [64, 64, 128],       
    "128_128":           [128, 128]
    }

architecture_results = []

for name, channels in architectures.items():
    config = VAEConfig(encoder_channels=channels, epochs=40, latent_dimension=1024, kl_weight=0.01, batch_size=32)
    architecture_results.append(run_and_evaluate_configuration(f"Arch_{name}", config, train_dataset))

all_experiment_results.append(architecture_results)
display_results(architecture_results, "+ Architecture")

100%|██████████| 613/613 [01:34<00:00,  6.50it/s]


,name,fid,latent_dimension,encoder_channels,batch_size,learning_rate,kl_weight,epochs
0,Arch_16_32_64,171.18,1024,"[16, 32, 64]",32,1.0e-03,0.01,40
1,Arch_64_64_128,180.67,1024,"[64, 64, 128]",32,1.0e-03,0.01,40
2,Arch_64_128_256,185.62,1024,"[64, 128, 256]",32,1.0e-03,0.01,40
3,Arch_16_32_64_128,190.95,1024,"[16, 32, 64, 128]",32,1.0e-03,0.01,40
4,Arch_64_128,194.93,1024,"[64, 128]",32,1.0e-03,0.01,40
5,Arch_128_128,212.79,1024,"[128, 128]",32,1.0e-03,0.01,40
6,Arch_128_256,214.08,1024,"[128, 256]",32,1.0e-03,0.01,40
7,Arch_32_64_128_256,476.15,1024,"[32, 64, 128, 256]",32,1.0e-03,0.01,40


##### Basic Hyperparameters

In [58]:
hyperparameters_experiments = []
hyperparameter_configs = [
    ("Batch32_LR1e-3", VAEConfig(batch_size=32, learning_rate=1e-3, epochs=40, latent_dimension=1024, kl_weight=0.01)),
    ("Batch64_LR1e-3", VAEConfig(batch_size=64, learning_rate=1e-3, epochs=40, latent_dimension=1024, kl_weight=0.01)),
    ("Batch128_LR1e-3", VAEConfig(batch_size=128, learning_rate=1e-3, epochs=40, latent_dimension=1024, kl_weight=0.01)),
    ("Batch64_LR3e-4", VAEConfig(batch_size=64, learning_rate=3e-4, epochs=40, latent_dimension=1024, kl_weight=0.01)),
    ("Batch32_LR3e-4", VAEConfig(batch_size=32, learning_rate=3e-4, epochs=40, latent_dimension=1024, kl_weight=0.01)),
]
for name, config in hyperparameter_configs:
    hyperparameters_experiments.append(run_and_evaluate_configuration(name, config, train_dataset))

all_experiment_results.append(hyperparameters_experiments)
display_results(hyperparameters_experiments, "+ Hyperparameters")


100%|██████████| 613/613 [01:33<00:00,  6.52it/s]


,name,fid,latent_dimension,encoder_channels,batch_size,learning_rate,kl_weight,epochs
0,Batch128_LR1e-3,172.44,1024,"[32, 64, 128]",128,1.0e-03,0.01,40
1,Batch64_LR1e-3,173.41,1024,"[32, 64, 128]",64,1.0e-03,0.01,40
2,Batch32_LR1e-3,173.45,1024,"[32, 64, 128]",32,1.0e-03,0.01,40
3,Batch32_LR3e-4,181.83,1024,"[32, 64, 128]",32,3.0e-04,0.01,40
4,Batch64_LR3e-4,186.66,1024,"[32, 64, 128]",64,3.0e-04,0.01,40


##### Combined Best Configurations

In [59]:
# ???

### Summary and Export

In [60]:
summary_df = pd.DataFrame(all_experiment_results).sort_values("fid")
print("\n=== FINAL EXPERIMENT SUMMARY ===")
print(summary_df[["name", "fid"]].round(2))

summary_df.to_csv("vae_experiments_summary.csv", index=False)

# Clean up flat reference view
if os.path.exists(FLAT_REFERENCE_DIR):
    shutil.rmtree(FLAT_REFERENCE_DIR)

AttributeError: 'list' object has no attribute 'keys'

In [ ]:
final_best_config = VAEConfig(
    latent_dimension=1024,
    encoder_channels=[32, 64, 128, 256],
    batch_size=64,
    kl_weight=0.01,
    epochs=100,
    learning_rate=1e-3
)

final_experiment = VAEExperiment(final_best_config)
final_experiment.fit(create_data_loader(train_dataset, final_best_config))

final_samples = denormalize_images(final_experiment.generate_samples(1000), final_best_config)
torch.save(final_samples.cpu().detach(), "piatek_Biegacz_Cieslik.pt")
print("Process complete. Submission file saved as: piatek_Biegacz_Cieslik.pt")